# Домашнее задание 7. Сборка конвейера CI/CD
Если у вас еще нет аккаунта в GitLab, вам нужно будет его создать:
1. Перейдите на [GitLab](https://gitlab.com/) и войдите в свой аккаунт.
2. Нажмите на кнопку New Project (Новый проект).
3. Выберите Create blank project (Создать пустой проект).
4. Укажите имя проекта и описание (по желанию).
5. Выберите уровень видимости проекта (Public).
6. Нажмите Create project (Создать проект).
7. Дополните файл .gitlab-ci.yml необходимыми джобами и отправьте в репозиторий.

## 1. Настроить CI/CD-пайплайн для ML-сервиса с использованием GitLab




Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

Вам дан рабочий код пайплайна и черновик файла .gitlab-ci.yml. Перепишите yaml в [ячейке](#scrollTo=s55MrS66JXWs)


*Ожидаемый артефакт: список коммитов в [ячейке](#scrollTo=gErasBmRSHjb) и ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=F0uQqbe3iHqE)*    

In [1]:
# репозиторий уже создан, тут только ставлю зависимости для проверки
!python3 --version
!python3 -m pip install -r requirements.txt -r requirements-dev.txt -q


Python 3.12.3


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [2]:
# вынес обучение в отдельный файл, чтобы pipeline был повторяемый
!sed -n '1,220p' ml_pipeline.py


from __future__ import annotations

import argparse
import json
from pathlib import Path

import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


def train_model(model_out: Path, metrics_out: Path, random_state: int = 42) -> dict[str, float | str]:
    iris = load_iris()
    x_train, x_test, y_train, y_test = train_test_split(
        iris.data,
        iris.target,
        test_size=0.2,
        random_state=random_state,
        stratify=iris.target,
    )

    model = RandomForestClassifier(n_estimators=100, random_state=random_state)
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    accuracy = accuracy_score(y_test, y_pred)

    model_out.parent.mkdir(parents=True, exist_ok=True)
    metrics_out.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, model_out)

    metrics = {
        "dataset": "sklearn.da

### Проверяем работоспособность пайплайна

In [3]:
!python3 ml_pipeline.py --model-out models/model.joblib --metrics-out reports/offline_metrics.json
!python3 model_gate.py --metrics reports/offline_metrics.json --policy policies/model_policy.yaml


accuracy=0.9000
model=models/model.joblib


model_gate: accuracy=0.9000, threshold=0.9000


In [4]:
# мой GitLab pipeline: проверки -> тесты -> обучение -> manifest -> план деплоя
!sed -n '1,240p' .gitlab-ci.yml


stages:
  - validate
  - test
  - train
  - package
  - deploy_plan

variables:
  PIP_CACHE_DIR: "$CI_PROJECT_DIR/.cache/pip"

cache:
  paths:
    - .cache/pip

default:
  image: python:3.11-slim
  before_script:
    - python -m pip install --upgrade pip

validate:
  stage: validate
  script:
    - pip install -r requirements.txt
    - python -m compileall -q app ml_pipeline.py model_gate.py
    - |
      python - <<'PY'
      import json
      from pathlib import Path
      required = ["Dockerfile", "docker-compose.blue.yml", "docker-compose.green.yml", "README.md"]
      missing = [name for name in required if not Path(name).exists()]
      if missing:
          raise SystemExit(f"missing required files: {missing}")
      print(json.dumps({"required_files": "ok"}, ensure_ascii=False))
      PY

unit_tests:
  stage: test
  script:
    - pip install -r requirements.txt -r requirements-dev.txt
    - pytest -q --maxfail=1

train_smoke:
  stage: train
  script:
    - pip install -r requir

In [5]:
# перед отправкой смотрю что реально поменялось в репозитории
!git status --short --branch
!git log --oneline -5


## main...origin/main


38b89e6 (HEAD -> main, origin/main) feat: complete HW7 ML deployment solution
b7defb8 Initial HW7 ML deployment solution


### Проверка статуса пайплайна

После настройки файла `.gitlab-ci.yml`, вы можете закоммитить изменения и запушить их в репозиторий.

GitLab автоматически запустит пайплайн, и вы сможете наблюдать за его выполнением в разделе CI/CD своего проекта.

Что нужно сделать:

1. Перейдите в свой проект на GitLab.
2. Нажмите на вкладку CI/CD и выберите Pipelines.
3. Вы увидите список запущенных пайплайнов. Нажмите на последний, чтобы увидеть выполнение.
4. Убедитесь, что все джобы выполнены успешно (отмечены зеленым цветом).
5. Приложите ссылку на статус выполнения в разделе Pipelines **своего** репозитория на GitLab.

**Ссылка на GitLab pipeline**

Я вынес GitLab pipeline в `.gitlab-ci.yml`. В нем не просто запуск скрипта, а несколько шагов:

1. `validate` - проверяю, что нужные файлы на месте
2. `unit_tests` - гоняю pytest
3. `train_smoke` - обучаю модель и проверяю model gate
4. `package_manifest` - проверяю Docker / compose файлы
5. `deploy_plan` - печатаю порядок Blue-Green релиза

Ссылка на запуск pipeline в GitLab:

```
<ссылка на мой GitLab pipeline после push в GitLab>
```

Токены и ключи в ноутбук не вставляю, потому что это сразу плохая практика.


## 2. Обосновать стратегию деплоя (развертывания, Blue-Green, Canary, Rolling, Shadow) и оценить влияние на риски




Изучите [инструмент](https://github.com/npryce/adr-tools) для учета архитектурных решений и запишите **причины**, по которым мы начали использовать стратегию деплоя и **риски**, к которым нас привело такое решение.



*Ожидаемый артефакт: архитектурное решение в формате ADR в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

**ADR / решение по деплою**

Я выбрал Blue-Green. Сначала хотел взять Canary, потому что он выглядит более "продовым": 90/10, потом больше трафика на новую версию и т.д. Но для этой работы Canary тяжело честно показать локально: нет нормального роутера с процентами и нет живого потока пользователей.

Что у меня получилось по Blue-Green:

1. blue - старая версия `v1.0.0`
2. green - новая версия `v1.1.0`
3. nginx смотрит на активную версию
4. если green сломался, я возвращаю трафик на blue через `scripts/rollback_blue.sh`

Риски:

- две версии сервиса работают одновременно, значит ресурсов надо больше
- если бы была база или сложные фичи, пришлось бы отдельно проверять совместимость схем
- сам Blue-Green не отвечает на вопрос "новая модель лучше для пользователей?", для этого уже нужен A/B тест

**Итог:** для сервиса без нормальной обработки ошибок Blue-Green спокойнее. Я могу сначала проверить green через `/health` и `/predict`, а только потом переключать трафик.


## 3. Реализовать стратегию развертывания

Реализуйте стратегию, выбранную на предыдущем [шаге](#scrollTo=hoQdM6SrJXXE).



*Ожидаемый артефакт: yaml в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

In [6]:
# реализация Blue-Green: два compose-файла и nginx proxy
!sed -n '1,180p' docker-compose.blue.yml
!sed -n '1,180p' docker-compose.green.yml
!sed -n '1,180p' docker-compose.proxy.yml
!sed -n '1,160p' scripts/switch_traffic.sh
!sed -n '1,160p' scripts/rollback_blue.sh


services:
  ml-blue:
    build:
      context: .
    image: ml-ops-hw7:v1.0.0
    container_name: ml-ops-hw7-blue
    environment:
      MODEL_VERSION: v1.0.0
    ports:
      - "8001:8000"
    healthcheck:
      test: ["CMD-SHELL", "python -c \"import urllib.request; urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=3)\""]
      interval: 10s
      timeout: 5s
      retries: 5


services:
  ml-green:
    build:
      context: .
    image: ml-ops-hw7:v1.1.0
    container_name: ml-ops-hw7-green
    environment:
      MODEL_VERSION: v1.1.0
    ports:
      - "8002:8000"
    healthcheck:
      test: ["CMD-SHELL", "python -c \"import urllib.request; urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=3)\""]
      interval: 10s
      timeout: 5s
      retries: 5


services:
  nginx:
    image: nginx:1.27-alpine
    container_name: ml-ops-hw7-proxy
    ports:
      - "18080:80"
    extra_hosts:
      - "host.docker.internal:host-gateway"
    volumes:
      - ./nginx/default.conf:/etc/nginx/conf.d/default.conf:ro


#!/usr/bin/env bash
set -euo pipefail

target="${1:-}"
if [[ "${target}" != "blue" && "${target}" != "green" ]]; then
  echo "usage: scripts/switch_traffic.sh blue|green" >&2
  exit 2
fi

cat "nginx/${target}.conf" > nginx/default.conf

if docker ps --format '{{.Names}}' | grep -q '^ml-ops-hw7-proxy$'; then
  docker compose -f docker-compose.proxy.yml exec nginx nginx -s reload
  sleep 1
fi

echo "traffic switched to ${target}"


#!/usr/bin/env bash
set -euo pipefail

scripts/switch_traffic.sh blue
scripts/smoke.sh http://127.0.0.1:18080


## 4. Спланировать A/B-тестирование для ML-модели

Вспомните материалы [семинара](https://colab.research.google.com/drive/1TM1yieSFhUqVxBferzbcexpAtK00lGYe?usp=sharing) и опишите параметры эксперимента.



*Ожидаемый артефакт: код в [ячейке](#scrollTo=OluzjqEhaIpM)*

In [7]:
# план A/B теста в виде словаря, чтобы было видно параметры эксперимента
ab_test_plan = {
    "variants": {
        "A": {"model_version": "v1.0.0", "traffic_share": 0.5},
        "B": {"model_version": "v1.1.0", "traffic_share": 0.5},
    },
    "assignment": "stable hash по user_id/session_id, чтобы пользователь не прыгал между вариантами",
    "primary_metric": "business conversion или accepted prediction rate после накопления labels",
    "technical_guardrails": {
        "p95_latency_ms_max": 500,
        "error_rate_max": 0.01,
        "offline_accuracy_min": 0.90,
    },
    "logged_fields": [
        "request_id", "user_id_hash", "variant", "model_version",
        "prediction", "latency_ms", "status_code", "timestamp"
    ],
    "stopping_rule": "фиксированное окно эксперимента + достаточный размер выборки",
}

ab_test_plan


{'variants': {'A': {'model_version': 'v1.0.0', 'traffic_share': 0.5},
  'B': {'model_version': 'v1.1.0', 'traffic_share': 0.5}},
 'assignment': 'stable hash по user_id/session_id, чтобы пользователь не прыгал между вариантами',
 'primary_metric': 'business conversion или accepted prediction rate после накопления labels',
 'technical_guardrails': {'p95_latency_ms_max': 500,
  'error_rate_max': 0.01,
  'offline_accuracy_min': 0.9},
 'logged_fields': ['request_id',
  'user_id_hash',
  'variant',
  'model_version',
  'prediction',
  'latency_ms',
  'status_code',
  'timestamp'],
 'stopping_rule': 'фиксированное окно эксперимента + достаточный размер выборки'}

## 5. Создать CI/CD-пайплайн для ML-сервиса с использованием GitHub Actions



*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*



Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

In [8]:
# файл обучения уже есть в репозитории, второй раз его не перезаписываю
!sed -n '1,220p' ml_pipeline.py


from __future__ import annotations

import argparse
import json
from pathlib import Path

import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


def train_model(model_out: Path, metrics_out: Path, random_state: int = 42) -> dict[str, float | str]:
    iris = load_iris()
    x_train, x_test, y_train, y_test = train_test_split(
        iris.data,
        iris.target,
        test_size=0.2,
        random_state=random_state,
        stratify=iris.target,
    )

    model = RandomForestClassifier(n_estimators=100, random_state=random_state)
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    accuracy = accuracy_score(y_test, y_pred)

    model_out.parent.mkdir(parents=True, exist_ok=True)
    metrics_out.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, model_out)

    metrics = {
        "dataset": "sklearn.da

Проверяем работоспособность пайплайна

In [9]:
!python3 ml_pipeline.py --model-out models/model.joblib --metrics-out reports/offline_metrics.json


accuracy=0.9000
model=models/model.joblib


Вам дан рабочий код пайплайна и черновик файла ci.yml. Используйте GitHub Actions и перепишите [шаг](#scrollTo=NGcDFbCFJXV_) name: Make pipeline reproducible

In [10]:
# GitHub Actions: отдельно CI и deploy workflow
!sed -n '1,220p' .github/workflows/ci.yml
!sed -n '1,260p' .github/workflows/deploy.yml


name: CI

on:
  pull_request:
  push:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt -r requirements-dev.txt

      - name: Static smoke
        run: python -m compileall -q app ml_pipeline.py model_gate.py

      - name: Unit tests
        run: pytest -q --maxfail=1

      - name: Train smoke and model gate
        run: |
          python ml_pipeline.py --model-out models/model.joblib --metrics-out reports/offline_metrics.json
          python model_gate.py --metrics reports/offline_metrics.json --policy policies/model_policy.yaml

      - uses: actions/upload-artifact@v4
        with:
          name: offline-metrics
          path: reports/offline_metrics.json


name: Model Deployment

on:
  push:
    branches: [main]
  workflow_dispatch:

permissions:
  contents: read
  packages: write

jobs:
  deploy:
    runs-on: ubuntu-latest
    env:
      IMAGE_NAME: ghcr.io/${{ github.repository }}
      MODEL_VERSION: ${{ vars.MODEL_VERSION || 'v1.1.0' }}

    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install and test
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt -r requirements-dev.txt
          pytest -q --maxfail=1
          python ml_pipeline.py --model-out models/model.joblib --metrics-out reports/offline_metrics.json
          python model_gate.py --metrics reports/offline_metrics.json --policy policies/model_policy.yaml

      - name: Build Docker image
        run: |
          docker build \
            --build-arg MODEL_VERSION="${MODEL_VERSION}" \
            -t "${IMAGE_NAME}:${{ github.

Копируем ci.yml в правильную директорию .github/workflows

In [11]:
# workflow-файлы уже лежат в .github/workflows
!find .github/workflows -maxdepth 1 -type f -print -exec sed -n '1,40p' {} \;


.github/workflows/ci.yml
name: CI

on:
  pull_request:
  push:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt -r requirements-dev.txt

      - name: Static smoke
        run: python -m compileall -q app ml_pipeline.py model_gate.py

      - name: Unit tests
        run: pytest -q --maxfail=1

      - name: Train smoke and model gate
        run: |
          python ml_pipeline.py --model-out models/model.joblib --metrics-out reports/offline_metrics.json
          python model_gate.py --metrics reports/offline_metrics.json --policy policies/model_policy.yaml

      - uses: actions/upload-artifact@v4
        with:
          name: offline-metrics
          path: reports/offline_metrics.json
.github/workflows

Начинаем отправку в репозиторий

In [12]:
# перед push проверяю состояние ветки
!git status --short --branch


## main...origin/main


После настройки workflow каждый раз при пуше в репозиторий GitHub Actions будет автоматически запускать конвейер. Пожалуйста, приложите ссылку на статус выполнения в разделе Actions **своего** репозитория на GitHub.


*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*

**GitHub Actions**

Для GitHub я сделал два workflow:

1. `CI` - ставит зависимости, запускает pytest, обучает модель и проверяет policy
2. `Model Deployment` - собирает Docker image, пушит его в GHCR и потом делает deploy через API, если есть секреты

Мне не хотелось оставлять фейковый `curl https://api.cloudprovider/deploy`, потому что workflow тогда падал бы без настоящего cloud endpoint. Поэтому я сделал условие: если `CLOUD_TOKEN` и `DEPLOY_API_URL` не заданы, внешний deploy шаг пропускается, но сборка Docker image все равно проверяется.

Ссылка на запуск GitHub Actions:

```
<ссылка на мой GitHub Actions run после push>
```


## 6. Итоговое оформление

В итоговых выводах дайте 5–8 предложений о своем опыте работы с инструментами модуля: что оказалось простым, что вызвало трудности, какие выводы сделали по обоснованию стратегии деплоя.

**Мой итог:**

1. Сначала казалось, что CI/CD - это просто yaml, который запускает `python ml_pipeline.py`, но по факту для ML-сервиса этого мало.
2. Пришлось отдельно вынести обучение, model gate, тесты API и Docker-сборку, чтобы pipeline был повторяемый.
3. Самое странное место для меня было с деплоем: в задании есть пример с внешним API, но реального cloud endpoint у меня нет, поэтому я сделал безопасный conditional step.
4. Blue-Green оказался понятнее Canary: две версии подняты рядом, проверил green, переключил nginx, при проблеме вернул blue.
5. A/B тест я воспринимаю уже как следующий слой после релиза: деплой отвечает "как безопасно выкатить", а A/B - "стала ли модель лучше для пользователей".
6. В итоге стало понятнее, почему для ML-модели нельзя просто заменить контейнер и радоваться: надо смотреть метрики, latency, ошибки и иметь быстрый rollback.

Основные файлы, которые я использовал:

- `.gitlab-ci.yml`
- `.github/workflows/ci.yml`
- `.github/workflows/deploy.yml`
- `Dockerfile`
- `docker-compose.blue.yml`
- `docker-compose.green.yml`
- `docker-compose.proxy.yml`
- `doc/architecture/decisions/0001-use-blue-green-deployment.md`


In [13]:
# локальный smoke, Docker должен быть запущен
!python3 -m compileall -q app ml_pipeline.py model_gate.py
!pytest -q
# !scripts/local_blue_green_smoke.sh


...

.

.                                                                    [100%]
5 passed in 1.04s
